In [6]:

# ============================================================
# CUSTOMER FEEDBACK ANALYSIS SYSTEM
# Existing: Bag of Words + TF-IDF
# Added: Word2Vec
# ============================================================

# If gensim is not installed, run this in a separate cell:
# %pip install gensim

import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from gensim.models import Word2Vec

# ============================================================
# 1. DATA COLLECTION
# ============================================================

data = {
    "review": [
        "I REALLY loved this Phone!!! Best camera ever 😍😍 Visit https://bit.ly/x",
        "Worst phone EVER!!! 😡😡 Battery drains in 2 hours... Don't buy!!!",
        "Camera quality is AMAZING!!! 😊😊 Very happy with purchase. Visit www.shop.com",
        "The phone is okay... Nothing special 🤔"
    ],
    "sentiment": ["Positive", "Negative", "Positive", "Neutral"]
}

customer_review = pd.DataFrame(data)

print("Original Reviews:")
print(customer_review)

# ============================================================
# 2. DATA PREPARATION
# ============================================================

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

customer_review["clean_review"] = customer_review["review"].apply(preprocess)

print("\nCleaned Reviews:")
print(customer_review[["review", "clean_review"]])

# ============================================================
# 3. BAG OF WORDS
# ============================================================

print("\n========== BAG OF WORDS ==========")

cv = CountVectorizer(ngram_range=(1, 2))
X_bow = cv.fit_transform(customer_review["clean_review"])

bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=cv.get_feature_names_out()
)

print(bow_df)

# ============================================================
# 4. TF-IDF VECTORIZATION
# ============================================================

print("\n========== TF-IDF ==========")

tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(customer_review["clean_review"])

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

print(tfidf_df.round(2))

# ============================================================
# 5. WORD2VEC MODEL - NEW ADDITION
# ============================================================

print("\n========== WORD2VEC ==========")

# Convert cleaned reviews into lists of words.
tokenized_reviews = [
    review.split()
    for review in customer_review["clean_review"]
]

# Train Word2Vec on the available customer reviews.
word2vec_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    seed=42,
    sg=1
)

# Convert each review into a fixed-size vector by averaging
# the Word2Vec vectors of its words.
def get_word2vec_vector(text, model, vector_size=100):
    tokens = text.split()

    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return [0.0] * vector_size

    return sum(vectors) / len(vectors)

X_word2vec = [
    get_word2vec_vector(
        review,
        word2vec_model,
        vector_size=100
    )
    for review in customer_review["clean_review"]
]

X_word2vec = pd.DataFrame(X_word2vec).values

word2vec_df = pd.DataFrame(
    X_word2vec,
    columns=[f"word2vec_{i}" for i in range(100)]
)

print("Word2Vec feature matrix shape:", X_word2vec.shape)
print(word2vec_df.round(3))

# ============================================================
# 6. MODEL BUILDING AND TRAINING
# ============================================================

y = customer_review["sentiment"]

# Model 1: Logistic Regression + Bag of Words
bow_model = LogisticRegression(max_iter=1000)
bow_model.fit(X_bow, y)

# Model 2: Logistic Regression + TF-IDF
tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_tfidf, y)

# Model 3: Logistic Regression + Word2Vec
word2vec_classifier = LogisticRegression(max_iter=1000)
word2vec_classifier.fit(X_word2vec, y)

print("\nAll three models have been trained.")

# ============================================================
# 7. MODEL TESTING AND PREDICTION
# ============================================================

new_review = ["Bad camera and battery"]

new_review_clean = [
    preprocess(text)
    for text in new_review
]

# Prediction using Bag of Words
new_review_bow = cv.transform(new_review_clean)
bow_prediction = bow_model.predict(new_review_bow)

# Prediction using TF-IDF
new_review_tfidf = tfidf.transform(new_review_clean)
tfidf_prediction = tfidf_model.predict(new_review_tfidf)

# Prediction using Word2Vec
new_review_word2vec = [
    get_word2vec_vector(
        text,
        word2vec_model,
        vector_size=100
    )
    for text in new_review_clean
]

word2vec_prediction = word2vec_classifier.predict(
    new_review_word2vec
)

print("\n========== PREDICTION RESULTS ==========")
print("New Review:", new_review[0])
print("Cleaned Review:", new_review_clean[0])

print("Bag of Words Prediction:", bow_prediction[0])
print("TF-IDF Prediction:", tfidf_prediction[0])
print("Word2Vec Prediction:", word2vec_prediction[0])


Original Reviews:
                                              review sentiment
0  I REALLY loved this Phone!!! Best camera ever ...  Positive
1  Worst phone EVER!!! 😡😡 Battery drains in 2 hou...  Negative
2  Camera quality is AMAZING!!! 😊😊 Very happy wit...  Positive
3             The phone is okay... Nothing special 🤔   Neutral

Cleaned Reviews:
                                              review  \
0  I REALLY loved this Phone!!! Best camera ever ...   
1  Worst phone EVER!!! 😡😡 Battery drains in 2 hou...   
2  Camera quality is AMAZING!!! 😊😊 Very happy wit...   
3             The phone is okay... Nothing special 🤔   

                                     clean_review  
0       really loved phone best camera ever visit  
1  worst phone ever battery drain 2 hour dont buy  
2     camera quality amazing happy purchase visit  
3                      phone okay nothing special  

========== BAG OF WORDS ==========
   amazing  amazing happy  battery  battery drain  best  best camera  bu

In [3]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 55.2 MB/s eta 0:00:00
